### Phase 4: Backtesting-Strategie
Wir plotten zunächst die Regime-Wahrscheinlichkeiten der Modelle sowie der tatsächlichen Modell-Signale.

Anschließend testen wir den Erfolg einer Investition in Abhängigkeit zum gewählten Modell und den unten beschriebenen Regeln.

*   **Regel:**
    *   Wenn Modell sagt "Bull": 100% Aktien (S&P 500).
    *   Wenn Modell sagt "Bear": 100% Bonds oder Cash.
*   **Transaktionskosten:** Integriere realistische Kosten (z.B. 0,1% pro Trade), da ML-Modelle oft zu nervös hin- und herschalten ("Churning").

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
import pandas as pd

# Daten aus dem data-Ordner laden
test_df = pd.read_parquet(cfg.data_path("test_data"))

In [ ]:
# --- 5. Dynamisches Backtesting (Automatisierter Vergleich aller Modelle) ---

import matplotlib.pyplot as plt
import numpy as np

# Parameterdefinition aus zentraler Config
transaction_costs = cfg.transaction_cost_rate  # z.B. 10 bps → 0.001

def backtest(df, signal_col, fee=None):
    """
    Berechnet die kumulierte Rendite unter Berücksichtigung von Transaktionskosten.
    fee: Kosten für einen vollständigen Wechsel (z.B. 0.1% = 0.001)
    """
    if fee is None:
        fee = cfg.transaction_cost_rate

    # Signal um konfigurierbare Tage verschieben zur Vermeidung von Look-ahead Bias
    trading_signal = df[signal_col].shift(cfg.backtesting.signal_shift).fillna(0)
    
    # Trades identifizieren: Wo unterscheidet sich das Signal von heute zu gestern?
    trades = trading_signal.diff().fillna(0).abs()
    
    # Logik: Wenn Signal 0 -> Portfolio-Return, sonst Cash-Return
    strategy_returns = np.where(trading_signal == 0, 
                                df['Returns'], 
                                df['Cash_Returns'])
    
    # Transaktionskosten abziehen
    net_strategy_returns = strategy_returns - (trades * fee)
    
    # Kumulierte Rendite berechnen
    return (1 + net_strategy_returns).cumprod()

# 1. Alle verfügbaren Modelle dynamisch identifizieren (anhand der _Signal Endung)
signal_cols = [col for col in test_df.columns if col.endswith('_Signal')]

# 2. Ergebnisse-DataFrame initialisieren
backtesting_results = pd.DataFrame(index=test_df.index)
# DataFrame für den zeitlichen Verlauf der Transaktionskosten
backtesting_transaction_costs = pd.DataFrame(index=test_df.index)

# Benchmark berechnen (Buy & Hold des 60/40 Portfolios)
backtesting_results['Buy_Hold'] = (1 + test_df['Returns']).cumprod()
# Buy & Hold hat 0 Transaktionskosten, da wir nie umschichten
backtesting_transaction_costs['Buy_Hold'] = 0.0

# Alle erkannten Modelle dynamisch backtesten
for sig_col in signal_cols:
    model_name = sig_col.rsplit('_', 1)[0]
    
    print(f"Berechne Backtest für {model_name} mit {transaction_costs*100}% Kosten...")
    backtesting_results[model_name] = backtest(test_df, sig_col, fee=transaction_costs)
    
    # Transaktionskosten im zeitlichen Verlauf berechnen
    trading_signal = test_df[sig_col].shift(cfg.backtesting.signal_shift).fillna(0)
    trades = trading_signal.diff().fillna(0).abs()
    backtesting_transaction_costs[model_name] = (trades * transaction_costs).cumsum()

# 3. Dynamische Visualisierung
plt.figure(figsize=(14, 7))

color_map = {
    'Buy_Hold': 'gray',
    'MS_Univariate': 'tab:blue',
    'MS_Exo': 'tab:red',
    'HMM': 'tab:purple',
    'LSTM': 'tab:green'
}

plt.plot(backtesting_results['Buy_Hold'], 
         label='Statisches 60/40 Portfolio (Benchmark)', 
         color=color_map.get('Buy_Hold', 'gray'), 
         alpha=0.5, linestyle='--')

for col in backtesting_results.columns:
    if col == 'Buy_Hold':
        continue
    
    color = color_map.get(col, None)
    linewidth = 2.5 if col == 'LSTM' else 1.5
    alpha = 1.0 if col == 'LSTM' else 0.8
    
    plt.plot(backtesting_results[col], 
             label=f'Strategie: {col.replace("_", " ")}', 
             color=color, linewidth=linewidth, alpha=alpha)

plt.title("Equity Curves: Dynamischer Vergleich der Regime-Switching-Modelle")
plt.xlabel("Datum")
plt.ylabel("Kumuliertes Vermögen (Start = 1.0)")
plt.legend(loc='upper left', ncol=2)
plt.grid(True, alpha=0.2)

# Equity Curve persistieren
plt.savefig(cfg.asset_path("equity_curves"), dpi=300, bbox_inches='tight')
plt.show()

print(f"Backtesting abgeschlossen. {len(signal_cols)} Strategien wurden berechnet.")

print("backtesting_results Dataframe")
print(backtesting_results)
print("backtesting_transaction_costs Dataframe")
print(backtesting_transaction_costs)

#--- Wir erhalten in diesem Schritt neben df und test_df backtesting_results mit kumulierten Werten ----
#--- Außerdem erhalten wir das Dataframe backtesting_transaction_costs mit Verläufen der Transaktionskosten ---

In [ ]:
# --- Performance & Drawdown Zusammenfassung ---

print("\n--- PERFORMANCE & DRAWDOWN ZUSAMMENFASSUNG ---")

summary_stats = []

for col in backtesting_results.columns:
    series = backtesting_results[col]
    
    # Finale Werte berechnen
    final_val = series.iloc[-1]
    total_ret = (final_val - 1) * 100
    
    # Max Drawdown berechnen
    roll_max = series.cummax()
    drawdown = series / roll_max - 1.0
    mdd = drawdown.min() * 100
    
    summary_stats.append({
        'Strategie': col,
        'Final Wealth': f"{final_val:.4f}",
        'Total Return': f"{total_ret:+.2f}%",
        'Max Drawdown': f"{mdd:.2f}%"
    })

# In DataFrame umwandeln
performance_summary_df = pd.DataFrame(summary_stats).set_index('Strategie')

# Als Markdown persistieren
performance_summary_df.to_markdown(cfg.asset_path("performance_summary"))

print("Zusammenfassung erfolgreich unter " + cfg.asset_path("performance_summary") + " gespeichert.")

# Im Notebook anzeigen
display(performance_summary_df)

In [ ]:
# Visualisierung der kumulierten Transaktionskosten
plt.figure(figsize=(14, 5))
for col in backtesting_transaction_costs.columns:
    if col == 'Buy_Hold': continue
    color = color_map.get(col, None)
    plt.plot(backtesting_transaction_costs[col] * 100, label=f'Kosten: {col.replace("_", " ")}', color=color)

plt.title(f"Kumulierte Transaktionskosten im Zeitverlauf (Gebühr: {transaction_costs*100}%)")
plt.ylabel("Kosten in %")
plt.legend(loc='upper left')
plt.grid(True, alpha=0.2)
plt.savefig(cfg.asset_path("transaction_costs"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#--- SORR Simulation: Analyse der Entnahmephase ---

def run_sorr_simulation(backtest_res, test_data, start_capital, monthly_withdrawal, fee_rate):
    """
    Führt eine SORR-Simulation für alle im DataFrame enthaltenen Strategien durch.
    """
    daily_ret = backtest_res.pct_change().fillna(0)
    path_results = pd.DataFrame(index=daily_ret.index)
    
    for col in daily_ret.columns:
        capital_path = []
        current_capital = start_capital
        last_month = -1
        
        # Signalzuordnung für Gebührenlogik
        sig_col = None
        if col == 'Buy_Hold': sig_col = None
        elif f"{col}_Signal" in test_data.columns: sig_col = f"{col}_Signal"
        elif col == 'HMM_Based' and 'HMM_Regime' in test_data.columns: sig_col = 'HMM_Regime'

        for date, ret in daily_ret[col].items():
            current_capital *= (1 + ret)
            
            if date.month != last_month:
                withdrawal = monthly_withdrawal
                is_invested = True
                if sig_col is not None and date in test_data.index:
                    if test_data.at[date, sig_col] == 1:
                        is_invested = False
                
                if is_invested:
                    withdrawal += (monthly_withdrawal * fee_rate)
                
                current_capital -= withdrawal
                last_month = date.month
            
            current_capital = max(0, current_capital)
            capital_path.append(current_capital)
            
        path_results[col] = capital_path
    return path_results

# Definition der Szenarien aus zentraler Config
scenarios = {}
for scenario_name, scenario_cfg in vars(cfg.backtesting.sorr.scenarios).items():
    scenarios[scenario_name] = {
        "start": scenario_cfg.initial_capital,
        "withdrawal": scenario_cfg.initial_capital * scenario_cfg.annual_withdrawal_rate / 12,
        "fee": scenario_cfg.liquidity_fee
    }

sorr_summaries = []

# Initialisierung eines leeren DataFrames für alle Simulationspfade
backtesting_sorr_simulation = pd.DataFrame(index=backtesting_results.index)

for name, params in scenarios.items():
    print(f"Running Scenario: {name}...")
    
    # Simulation berechnen
    sim_results = run_sorr_simulation(
        backtesting_results, 
        test_df, 
        params["start"], 
        params["withdrawal"], 
        params["fee"]
    )
    
    # Alle Ergebnisse dem zentralen DataFrame hinzufügen
    for col in sim_results.columns:
        column_name = f"{name}_{col}"
        backtesting_sorr_simulation[column_name] = sim_results[col]
    
    # Visualisierung pro Szenario
    plt.figure(figsize=(12, 5))
    for col in sim_results.columns:
        plt.plot(sim_results[col], label=col.replace('_', ' '))
    
    plt.title(f"SORR Szenario {name}: Start {params['start']:,.0f}€, Entnahme {params['withdrawal']:,.0f}€")
    plt.axhline(y=0, color='black', linestyle='-')
    plt.legend(loc='upper left', fontsize='small')
    plt.savefig(cfg.asset_path(f"sorr_sim_{name.lower()}"), dpi=300, bbox_inches='tight')
    plt.show()

    # Statistische Auswertung für dieses Szenario sammeln
    for col in sim_results.columns:
        final_cap = sim_results[col].iloc[-1]
        status = "Kapitalerhalt" if final_cap > 0 else f"Erschöpft ({sim_results[sim_results[col] <= 0].index[0].strftime('%Y')})"
        sorr_summaries.append({
            "Szenario": name,
            "Strategie": col.replace('_', ' '),
            "Endkapital": f"{final_cap:,.2f} €",
            "Status": status
        })

# 2.) Zusammenfassende Tabelle aller Szenarien erstellen und persistieren
sorr_multi_eval = pd.DataFrame(sorr_summaries).set_index(['Szenario', 'Strategie'])
sorr_multi_eval.to_markdown(cfg.asset_path("sorr_summary"), index=True)

print("Alle Szenarien berechnet und unter " + cfg.paths.assets_dir + "/ gespeichert.")
display(sorr_multi_eval)

print(backtesting_sorr_simulation)

#--- Wir erhalten in diesem Schritt neben df, test_df, backtesting_results, backtesting_transaction_costs nun backtesting_sorr_simulation ----

In [ ]:
output_path = cfg.data_path('backtesting_results')

# Speichern als Parquet
backtesting_results.to_parquet(output_path)

print(f"Dataframe erfolgreich unter {output_path} gespeichert.")

In [ ]:
output_path = cfg.data_path("backtesting_costs")

# Speichern als Parquet
backtesting_transaction_costs.to_parquet(output_path)

print(f"Transaktionskosten-Dataframe erfolgreich unter {output_path} gespeichert.")

In [ ]:
output_path = cfg.data_path("backtesting_sorr")

# Speichern als Parquet
backtesting_sorr_simulation.to_parquet(output_path)

print(f"SORR-Simulations-Dataframe erfolgreich unter {output_path} gespeichert.")